In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
import requests

TOKEN = os.getenv("TOKEN_NANO_LAB_ASSESSMENTS_5484") or "29839590236E10D1A025F2180A12C70C"
API_URL = os.getenv("REDCAP_API_URL", "https://redcap.research.sc.edu/api/")

def redcap_post(content, **params):
    payload = {"token": TOKEN, "content": content, "format": "json", "returnFormat": "json"}
    payload.update(params)
    r = requests.post(API_URL, data=payload, timeout=60)
    r.raise_for_status()
    return r.json()

print("Token loaded:", bool(TOKEN))
print("API URL:", API_URL)


In [ ]:
metadata = redcap_post("metadata")
status_field = next((f for f in metadata if f.get("field_name") == "demo_status"), None)
print("demo_status field metadata:")
print(status_field.get("select_choices_or_calculations"))

status_choice_map = {}
for item in (status_field or {}).get("select_choices_or_calculations", "").split("|"):
    match = re.match(r"\s*(\d+)\s*,\s*(.+)", item)
    if match:
        code, label = match.groups()
        status_choice_map[f"demo_status___{code}"] = label.strip()

print("Detected checkbox map:", status_choice_map)

score_fields = [
    f["field_name"] for f in metadata
    if f.get("field_name", "").startswith("csbsbs_") and "composite" in f.get("field_name", "") and "calc" in f.get("field_name", "")
]
print("CSBS score fields:", score_fields)


In [ ]:
records = redcap_post("record", action="export", format="json", type="flat", rawOrLabel="raw")
df = pd.DataFrame(records)
print("Raw records loaded:", df.shape)

status_cols = [c for c in df.columns if c.startswith("demo_status___")]

def normalize_group(label):
    if pd.isna(label):
        return "Other/Unknown"
    text = re.sub(r"[^A-Z0-9]+", " ", str(label).upper()).strip()
    if "ASIB" in text or "ASD" in text:
        return "ASIB"
    if "TD" in text or "TYPICAL" in text:
        return "TD"
    if "PT" in text or "PRETERM" in text:
        return "PT"
    return "Other/Unknown"

def cohort_from_status(row):
    for col in status_cols:
        if str(row.get(col, 0)).strip() == "1":
            label = status_choice_map.get(col, "")
            return normalize_group(label)
    return "Other/Unknown"

df_status = df[["record_id"] + status_cols].copy()
df_status = df_status.dropna(subset=status_cols, how="all").copy()
for col in status_cols:
    df_status[col] = pd.to_numeric(df_status[col], errors="coerce").fillna(0).astype(int)
df_status["Diagnostic Group"] = df_status.apply(cohort_from_status, axis=1)
df_status = df_status[["record_id", "Diagnostic Group"]].drop_duplicates(subset=["record_id"])

examiner_field = "csbsbs_examiner"
score_cols = [
    "csbsbs_socialcompositecalc",
    "csbsbs_speechcompositecalc",
    "csbsbs_symboliccompositecalc"
]

df_scores = df[["record_id", "redcap_event_name", examiner_field] + score_cols].copy()
for col in score_cols:
    df_scores[col] = pd.to_numeric(df_scores[col], errors="coerce")
df_scores = df_scores.merge(df_status, on="record_id", how="left")
df_scores = df_scores.dropna(subset=[examiner_field] + score_cols).copy()
df_scores["Diagnostic Group"] = df_scores["Diagnostic Group"].fillna("Other/Unknown")

def clean_visit(event_name):
    event = str(event_name or "").lower()
    if "24" in event:
        return "24 Months"
    if "18" in event:
        return "18 Months"
    if "15" in event:
        return "15 Months"
    if "12" in event:
        return "12 Months"
    if "9" in event:
        return "9 Months"
    if "6" in event:
        return "6 Months"
    if "3" in event:
        return "3 Months"
    if "2" in event:
        return "2 Months"
    return "Other"

df_scores["Study"] = "CSBS-BS"
df_scores["Visit"] = df_scores["redcap_event_name"].apply(clean_visit)
df_scores["Examiner"] = df_scores[examiner_field].astype(str).str.strip()
df_scores = df_scores[df_scores["Diagnostic Group"].isin(["ASIB", "TD", "PT"])].copy()

print("Grouped records retained:", df_scores.shape)
print(df_scores[["Examiner", "Study", "Visit", "Diagnostic Group", "csbsbs_socialcompositecalc"]].head().to_string(index=False))


In [ ]:
summary = (
    df_scores.groupby(["Examiner", "Study", "Visit", "Diagnostic Group"], as_index=False)
    .agg(
        Social_Mean=("csbsbs_socialcompositecalc", "mean"),
        Social_SD=("csbsbs_socialcompositecalc", "std"),
        Speech_Mean=("csbsbs_speechcompositecalc", "mean"),
        Speech_SD=("csbsbs_speechcompositecalc", "std"),
        Symbolic_Mean=("csbsbs_symboliccompositecalc", "mean"),
        Symbolic_SD=("csbsbs_symboliccompositecalc", "std"),
        N=("csbsbs_socialcompositecalc", "size"),
    )
    .sort_values(["Diagnostic Group", "Examiner", "Visit"])
    .reset_index(drop=True)
)

summary = summary[["Examiner", "Study", "Visit", "Diagnostic Group", "Social_Mean", "Social_SD", "Speech_Mean", "Speech_SD", "Symbolic_Mean", "Symbolic_SD", "N"]]
print("
--- Final workbook rows ---")
print(summary.head(20).to_string(index=False))

output_path = Path("/Users/namomac/esd-redcap-metadata-watcher/IPSA/csbsbs_metrics_by_examiner.xlsx")
summary.to_excel(output_path, index=False)
print(f"Saved workbook to {output_path}")
